# UMA Relaxation Showcase

This notebook demonstrates the **UMA (Universal Model for Atoms) relaxation** fidelity gate for catalyst discovery campaigns.

**IMPORTANT:** UMA/OMat24 energies are kept in a **SEPARATE TIER** from Materials Project energies due to different DFT settings (as documented in FAIRChem disclaimer). This showcase demonstrates the concept without mixing energy scales.

---

## Purpose

The UMA relaxation serves as a **final-fidelity sanity check** on the best candidate from a campaign (typically mp-2790, the lowest e_above_hull material). It answers:

1. Does the surrogate-selected candidate survive higher-fidelity optimization?
2. What is the structural relaxation effect on volume?
3. Is there a MACE-tier energy change after relaxation (both quantities computed by MACE, so directly comparable to each other -- never compared against the MP-tier e_above_hull shown separately for reference)?

**This is NOT for production use** -- it's a showcase demonstrating the fidelity gate concept. FAIRChem/UMA is an optional dependency; if it isn't installed, this notebook still runs and shows the MP-tier reference data, with the relaxation section clearly marked as skipped.


## Imports and Configuration

In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys_path_root = Path("__file__").resolve().parent.parent
if str(sys_path_root) not in sys.path:
    sys.path.insert(0, str(sys_path_root))

import json
import numpy as np
import matplotlib.pyplot as plt

from kg.graph_store import load_graph, rehydrate_node
from kg.schema import MaterialNode

# Configuration
KG_PATH = Path("data/processed/kg.json")
TARGET_MPID = "mp-2790"  # Best candidate from typical campaigns
# Matches agent/predictor.py's MACE_CHECKPOINT_PATH -- keep these in sync.
MACE_CHECKPOINT_PATH = Path("models/mace-mpa-0-medium.model")
FIG_SIZE = (10, 6)
DPI = 150

print("Imports complete")

## Load Target Material

In [ ]:
# Load KG
if not KG_PATH.exists():
    raise FileNotFoundError(f"KG file not found: {KG_PATH}")

G = load_graph(KG_PATH)

# Find target material
mat_nid = None
for nid, data in G.nodes(data=True):
    if data.get("type") == "Material" and data.get("mpid") == TARGET_MPID:
        mat_nid = nid
        break

if mat_nid is None:
    raise ValueError(f"Material {TARGET_MPID} not found in KG")

# Rehydrate with full structure info
mat = rehydrate_node(G, mat_nid)

print(f"Loaded: {mat.mpid}")
print(f"Formula: {mat.formula_pretty}")
print(f"Elements: {', '.join(mat.elements)}")

## Get MP Properties from KG

Extract the Materials Project-derived e_above_hull for reference (this is in the **MP TIER**).

In [ ]:
# Find property node
prop_nid = None
for nid, data in G.nodes(data=True):
    if (data.get("type") == "Property" and 
        data.get("mpid") == mat.mpid and
        data.get("name") == "energy_above_hull"):
        prop_nid = nid
        break

mp_eah = None
if prop_nid:
    props = G.nodes[prop_nid]
    mp_eah = float(props.get("value", np.nan))

if mp_eah is not None and not np.isnan(mp_eah):
    print(f"MP e_above_hull: {mp_eah:.4f} eV/atom")
    print(f"Stability status: {'STABLE' if mp_eah < 0.1 else 'UNSTABLE'}")
else:
    print(f"[WARN] No MP e_above_hull found in KG for {mat.mpid} -- reference value unavailable")
    mp_eah = None

## UMA Relaxation (if FAIRChem available)

Run the UMA relaxation on the target material. **This is a SEPARATE TIER** — do NOT mix with MP energies.

In [ ]:
# Load structure from CIF
struct_nid = mat.structure_id
cif_path = None
ase_atoms = None
pmg_struct = None

for nid, data in G.nodes(data=True):
    if nid == struct_nid and data.get("type") == "Structure":
        cif_path = Path(data.get("cif_path"))
        
        from pymatgen.core import Structure as PMGStructure
        pmg_struct = PMGStructure.from_file(str(cif_path))
        
        # Convert to ASE Atoms. pymatgen returns MSONAtoms, which IS an
        # ase.Atoms subclass -- do NOT rebuild it. Rebuilding without
        # pbc= silently drops periodic boundary conditions (ASE defaults
        # to pbc=False), and the crystal gets evaluated as an isolated
        # cluster in vacuum.
        ase_atoms = pmg_struct.to_ase_atoms()
        if not all(ase_atoms.get_pbc()):
            ase_atoms.set_pbc(True)
        
        print(f"Loaded structure: {len(ase_atoms)} atoms")
        print(f"Initial volume: {pmg_struct.volume:.2f} \u00c5\u00b3")
        break

if ase_atoms is None:
    raise ValueError(f"No Structure node found for {mat.mpid} (structure_id={struct_nid})")

# Try UMA relaxation if FAIRChem is available. This is optional-dependency
# code: if fairchem isn't installed, or its relaxation API doesn't match
# what's called here, this section is a SKIPPED demonstration of the
# fidelity-gate concept, not a verified result. Do not treat a
# "[INFO] skipped" outcome as a passed check.
uma_result = None

try:
    import fairchem
    from fairchem.core import pretrained_mlip, FAIRChemCalculator

    print("\nFAIRChem detected. Running UMA relaxation...")
    print("="*60)

    predictor = pretrained_mlip.get_predict_unit("uma-s-1", device="cpu")
    calc = FAIRChemCalculator(predictor, task_name="omat")
    ase_atoms.calc = calc

    from ase.optimize import FIRE
    dyn = FIRE(ase_atoms)
    dyn.run(fmax=0.05, steps=200)

    relaxed_atoms = ase_atoms
    relaxed_volume = float(relaxed_atoms.get_volume())
    uma_energy_per_atom = float(relaxed_atoms.get_potential_energy() / len(relaxed_atoms))

    print(f"Relaxation complete!")
    print(f"  Relaxed formula: {relaxed_atoms.get_chemical_formula()}")
    print(f"  Relaxed volume: {relaxed_volume:.2f} \u00c5\u00b3")
    print(f"  UMA-tier energy per atom (relaxed): {uma_energy_per_atom:.4f} eV/atom")

    uma_result = {
        "status": "completed",
        "relaxed_formula": relaxed_atoms.get_chemical_formula(),
        "relaxed_volume": relaxed_volume,
        "uma_energy_per_atom_eV": uma_energy_per_atom,
        # No cross-tier comparison here: UMA-tier energy is only ever
        # compared to another UMA-tier value, never to mp_eah (MP tier)
        # or to a MACE-tier value.
    }

except ImportError as e:
    print(f"[INFO] FAIRChem not available -- UMA relaxation skipped ({e})")
    uma_result = {"status": "skipped", "error": str(e)}

except Exception as e:
    print(f"[ERROR] UMA relaxation failed: {e}")
    import traceback
    traceback.print_exc()
    uma_result = {"status": "failed", "error": str(e)}

## Visualization

**IMPORTANT:** These plots show MP and UMA quantities as **SEPARATE TIER** values. Volume is compared directly (a geometric, not energetic, quantity, so tier-mixing does not apply). No cross-tier energy comparison is plotted. If the relaxation was skipped or failed, this cell reports that plainly instead of plotting invented data.

In [ ]:
if uma_result.get("status") != "completed":
    print(f"[INFO] UMA relaxation status: {uma_result.get('status')} -- nothing to plot.")
    print("Run this notebook in an environment with FAIRChem installed to see the relaxation plots.")
else:
    fig, ax1 = plt.subplots(1, 1, figsize=FIG_SIZE)

    mp_volume = pmg_struct.volume
    relaxed_volume = uma_result["relaxed_volume"]

    ax1.bar(['MP Structure', 'UMA Relaxed'], [mp_volume, relaxed_volume],
            color=['steelblue', 'coral'], alpha=0.7)
    ax1.set_ylabel('Volume (\u00c5\u00b3)')
    ax1.set_title(f'{mat.formula_pretty} Volume Comparison (MP structure vs UMA-relaxed)')

    for i, v in enumerate([mp_volume, relaxed_volume]):
        ax1.text(i, v + 0.1, f'{v:.2f}', ha='center', fontsize=9)

    plt.tight_layout()
    plots_dir = Path("notebooks/plots")
    plots_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(plots_dir / "uma_relaxation.png", dpi=DPI, bbox_inches='tight')
    print(f"\nSaved plot to {plots_dir / 'uma_relaxation.png'}")

## Results Summary

Save the UMA results as a **SEPARATE TIER** — do NOT mix with MP energies.

In [ ]:
# Compile results
results = {
    "material_id": mat.mpid,
    "formula": mat.formula_pretty,
    "elements": mat.elements,
    "mp_properties": {
        "e_above_hull_eV_per_atom": mp_eah,
        "stability_status": (
            "stable" if mp_eah is not None and mp_eah < 0.1
            else "unstable" if mp_eah is not None
            else "unknown"
        ),
    },
    "uma_relaxation": uma_result,
}

# Save results
output_dir = Path("models/verification")
output_dir.mkdir(parents=True, exist_ok=True)

results_file = output_dir / "uma_mp2790_results.json"
with open(results_file, 'w') as f:
    json.dump(results, f, indent=2, default=str)

print(f"\nSaved UMA results to {results_file}")

# Print summary
print("\n" + "="*60)
print("UMA Relaxation Showcase Summary")
print("="*60)
print(f"Target material: {mat.mpid} ({mat.formula_pretty})")
if mp_eah is not None:
    print(f"MP e_above_hull: {mp_eah:.4f} eV/atom")
    print(f"Stability status: {'STABLE' if mp_eah < 0.1 else 'UNSTABLE'}")
else:
    print("MP e_above_hull: not found in KG")

if uma_result.get("status") == "completed":
    ur = uma_result
    print(f"\nUMA relaxation: COMPLETED")
    print(f"  Relaxed formula: {ur.get('relaxed_formula', 'N/A')}")
    print(f"  Relaxed volume: {ur.get('relaxed_volume'):.2f} \u00c5\u00b3")
    print(f"  UMA-tier energy per atom: {ur.get('uma_energy_per_atom_eV'):.4f} eV/atom")
else:
    print(f"\nUMA relaxation: {uma_result.get('status', 'unknown').upper()}")
    if uma_result.get("error"):
        print(f"  Reason: {uma_result['error']}")

## Key Takeaways

### Energy Tier Separation

**IMPORTANT:** UMA/OMat24 energies are **NOT numerically compatible** with Materials Project energies, and MACE-surrogate energies (see `agent/predictor.py`) are a third, separate tier again -- all three use different levels of theory / training data. This showcase:

1. Shows the relaxation effect on structure (volume, geometry) -- a tier-agnostic geometric comparison
2. Demonstrates the fidelity gate concept
3. Keeps all energy values in **separate tiers**, with no cross-tier arithmetic anywhere in this notebook

**Do NOT mix MP, MACE, and UMA energies numerically** — they represent different computational frameworks.

---

### What This Demonstrates

- Volume changes after UMA relaxation (typically small for stable structures), when FAIRChem is available
- The shape of a final-fidelity fidelity-gate check, as a concept -- this notebook does NOT itself decide whether mp-2790 "survives" relaxation; no pass/fail criterion is computed here
- If FAIRChem is not installed, the notebook still runs end-to-end and reports the MP-tier reference data with the relaxation section clearly marked skipped, rather than failing or fabricating a result

---

### Next Steps

This is a **showcase** demonstrating the fidelity gate concept. For production use:

1. Define an explicit pass/fail criterion (e.g. volume change tolerance) if this is to gate anything automatically
2. Consider implementing as an optional final step in `campaign.py`
3. Add proper energy tier separation to KG schema (a `tier` or `source`-scoped query helper, since `source` already exists on PropertyNode)
4. Document the DFT setting differences clearly in README
